# TRAINING of Hyperparameters
### Three parameter as input case
This code as the purpose to find the best hyperparameters in the case 2 step. In particular, it is worth noticing that the first NN is already optimized. 

# DUBBIO SU HF E LF nel caso test, quali metto nella rete?

In [1]:
#########################     LIBRARIES     ##########################
import keras.backend as K
from keras.regularizers import l2
from hyperopt import STATUS_OK, tpe, Trials, hp, fmin
from hyperopt.pyll.stochastic import sample
from sklearn.model_selection import KFold
import numpy as np
from itertools import product
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import pyplot as plt
from matplotlib import cm
from matplotlib.ticker import LinearLocator, FormatStrFormatter
from keras.optimizers import Adam, Nadam, Adamax
from ann_functions3D import getModel, kCrossVal, transfBestparam, import_data
from time import perf_counter
import pandas
import pickle
import os

seed = 7

c:\Users\Giacomo\AppData\Local\Programs\Python\Python310\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.1
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [26]:
########################     PREPARATION      ##########################
# introduction of the data
file_path_LF = os.path.join("..", "..", "Diffusion\DATA", "reaction_diffusion_LF_46_d75.mat")
#file_path_LF = "..\..\Diffusion\DATA\reaction_diffusion_LF_46_d75.mat"
(reaction_LF_test, U_LF_test, x_LF_test) = import_data(file_path_LF)
file_path_HF = os.path.join("..", "..", "Diffusion\DATA", "reaction_diffusion_HF.mat")
(reaction_HF_test, U_HF_test, x_HF_test) = import_data(file_path_HF)

In [27]:
########################     NORMALIZATION  #########################
# Input
#reaction_LF_test = np.column_stack((reaction_LF_test, x_LF_test))
#reaction_HF_test = np.column_stack((reaction_HF_test, x_HF_test))


reaction_max = np.max(reaction_LF_test)
reaction_min = np.min(reaction_LF_test)

# reaction_test_norm = (reaction_test - reaction_min) / (reaction_max - reaction_min)
reaction_LF_test = (reaction_LF_test - reaction_min) / (
    reaction_max - reaction_min
)
reaction_HF_test = (reaction_HF_test - reaction_min) / (
    reaction_max - reaction_min
)

In [28]:
#########################     TRAIN SET      ##########################
NepoLF = 5000  # number of epochs for first NN: NN_LF
NepoLin = 1500       # number of epochs for second NN: NN_Lin

NepoHF = 3000  # number of epochs for second NN: NN_HF

permutation1 = np.random.permutation(len(reaction_LF_test))
permutation2 = np.random.permutation(len(x_LF_test))
Nlf = 30
reaction_LF = reaction_LF_test[permutation1][0:Nlf]
x_LF = x_LF_test[permutation2][0:Nlf]

reaction_LF = np.column_stack((reaction_LF, x_LF))
reaction_LF_test = np.array(list(product(reaction_LF_test.flatten(), x_LF_test.flatten())))

U_LF_test = U_LF_test[
    :,
     - 1,
    :,12
]

In [29]:
# TRANSFORMATION
U_t_max_test = np.max(U_LF_test)
U_t_min_test = np.min(U_LF_test)

U_LF_test = (U_LF_test - U_t_min_test) / (U_t_max_test - U_t_min_test)
#U_HF = (U_HF - U_h_min_test) / (U_h_max_test - U_h_min_test)
U_LF = U_LF_test[permutation1[0:Nlf],permutation2[0:Nlf]]
##
row, col = U_LF_test.shape
index_row, index_col = np.meshgrid(np.arange(row), np.arange(col), indexing='ij')
comb = np.ravel_multi_index((index_row.flatten(), index_col.flatten()), dims=(row, col))
U_LF_test = U_LF_test.flatten()[comb]
##
permutation1 = np.random.permutation(len(reaction_HF_test))
permutation2 = np.random.permutation(len(x_HF_test))
n_HF = 15
reaction_HF = reaction_HF_test[permutation1][0:n_HF]
x_HF = x_HF_test[permutation2][0:n_HF]
reaction_HF = np.column_stack((reaction_HF, x_HF))
U_HF_test = U_HF_test[:, -1, :,44]

reaction_HF_test = np.array(list(product(reaction_HF_test.flatten(), x_HF_test.flatten())))

# TRANSFORMATION
U_h_max_test = np.max(U_HF_test)
U_h_min_test = np.min(U_HF_test)
#U_HF = (U_HF - U_h_min_test) / (U_h_max_test - U_h_min_test)
U_HF_test = (U_HF_test - U_h_min_test) / (U_h_max_test - U_h_min_test)

##
U_HF = U_HF_test[permutation1[0:n_HF],permutation2[0:n_HF]]
row, col = U_HF_test.shape
index_row, index_col = np.meshgrid(np.arange(row), np.arange(col), indexing='ij')
comb = np.ravel_multi_index((index_row.flatten(), index_col.flatten()), dims=(row, col))
U_HF_test = U_HF_test.flatten()[comb]
##

In [18]:
##########################       FIRST NN: NN_LF     ##########################
K.clear_session()
bestLF_params = {
    "lr": 0.0255,
    "kernel_init": "glorot_uniform",
    "opt": "Adam",
}  # obtained

modelLF = getModel(bestLF_params, "LF")
histLF = modelLF.fit(
    reaction_LF, U_LF, epochs=NepoLF, batch_size=Nlf, verbose=0
)
print("LF NN done")

ULF = modelLF.predict(reaction_LF_test)
print("\nLF Model:")

test_mse = np.mean(np.square(U_LF_test - ULF[:, 0]))
print(f"Test MSE: {test_mse}")

r_2 = 1 - np.sum(np.square(U_LF_test - ULF[:, 0])) / np.sum(
    np.square(U_LF_test - np.mean(U_LF_test))
)
print(f"R^2: {r_2}")

LF NN done
72/72 [==============================] - 0s 4ms/step

LF Model:
Test MSE: 0.20894482848362536
R^2: -0.8245504703114273


In [31]:
start = perf_counter()

##########################    SECOND NN: NN_Lin    ##########################
hf_help = modelLF.predict(reaction_HF)
hf_lin = np.concatenate((reaction_HF, hf_help.reshape(-1,1)),axis=1)

K.clear_session()

test_help = modelLF.predict(reaction_HF_test)

bestLin_params = {'lr' : 0.001, 'kernel_init' : 'glorot_uniform', 'opt' : 'Adam', 'l2weight' : 0.01}
modelLin = getModel(bestLin_params,'Hflin')
histLin = modelLin.fit(hf_lin, U_HF, epochs= NepoLin, batch_size = n_HF, verbose = 0)

ULin = modelLin.predict(np.concatenate((reaction_HF_test, test_help), axis = 1))


157/157 [==============================] - 1s 4ms/step


In [35]:
##########################    THIRD NN: NN_HF    ########################## old
#Input for training NN_HF
hf_help1 = modelLin.predict(hf_lin)
hf_final = np.concatenate(
    (hf_lin, hf_help1),axis=1
)


#Input for testing NN_HF
reaction_train_help = modelLin.predict(np.concatenate((reaction_HF_test, test_help), axis = 1))
reaction_final = np.concatenate((reaction_HF_test, np.concatenate((test_help, reaction_train_help), axis=1)), axis=1)

name = '3step'

1/1 [==============================] - 0s 38ms/step


157/157 [==============================] - 0s 2ms/step


In [36]:
####################    HYPERPARAMETER OPTIMIZATION    #######################
MAX_EVAL = 1

K.clear_session()
bayes_trials = Trials()
opt_list = ["Adam", "Adamax"]
kernel_list = ["uniform", "glorot_uniform"]
aux_dic = {"opt": opt_list, "kernel_init": kernel_list}
space = {
    "nodes": hp.qloguniform("nodes", np.log(4), np.log(64), 2),
    "l2weight": hp.loguniform("l2weight", np.log(0.0001), np.log(100)),
    "lr": hp.loguniform("lr", np.log(0.0001), np.log(0.1)),
    "kernel_init": hp.choice("kernel_init", kernel_list),
    "opt": hp.choice("opt", opt_list),
}


def objective(params):
    K.clear_session()
    CVres = kCrossVal(2,n_HF, NepoHF, reaction_final, U_HF, params, name)
    # mse, r_squared = calculate_metrics(Nhf, NepoHF, mu_final, U_hf_train, params, name)   # ADDED

    # return {'loss': CVres, 'mse': mse, 'r_squared': r_squared, 'params': params, 'status': STATUS_OK}
    return {"loss": CVres, "params": params, "status": STATUS_OK}


best_params = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=MAX_EVAL,
    trials=bayes_trials,
)


  0%|          | 0/1 [00:00<?, ?trial/s, best loss=?]

job exception: index 15 is out of bounds for axis 0 with size 15



  0%|          | 0/1 [00:00<?, ?trial/s, best loss=?]


IndexError: index 15 is out of bounds for axis 0 with size 15

In [ ]:
transfBestparam(best_params, aux_dic)
print(best_params)

####################    NN_HF training and PREDICTION    #######################
finalModel = getModel(
    best_params, name
)  # final model chosen according to the best paramters
hist = finalModel.fit(
    reaction_final,
    U_HF,
    validation_data=(hf_final, U_HF_test),
    epochs=NepoHF,
    batch_size=n_HF,
    verbose=0,
    validation_freq=20,
)

UHF = finalModel.predict(hf_final)

stop = perf_counter()
elapsed = stop - start
print("Elapsed time: ", elapsed)
print("\nHF Model:")

test_mse = np.mean(np.square(U_HF_test - UHF[:, 0]))
print(f"Test MSE: {test_mse}")

r2_HF = 1 - np.sum(np.square(U_HF_test - UHF[:, 0])) / np.sum(
    np.square(U_HF_test - np.mean(U_HF_test))
)
print(f"R^2: {r2_HF}")

#print("Number of basis functions: ", int(Nlf_models[m]))
print("Number of HF data: ", n_HF)

In [ ]:
####################    TRAINING INSIGHTS    #######################
plt.figure()
plt.subplot(2, 1, 1)
plt.plot(hist.history["mse"], color="red", label="High fidelity train mse")
plt.plot(histLF.history["mse"], color="black", label="Low fidelity")
plt.legend()
plt.yscale("log")
plt.subplot(2, 1, 2)
plt.plot(hist.history["val_mse"], color="red")
plt.yscale("log")
plt.show()

In [ ]:
#################################BRUTTA##################################################
####################################################################################


import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

# Crea un grafico 3D
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

# Disegna i grafici di dispersione 3D e ottieni i Path3DCollection
scatter1 = ax.scatter(reaction_HF_test[:,0], reaction_HF_test[:,1], UHF, c='r', marker='o', label='estimated $U_{HF}$ by $NN_{HF}$')
scatter2 = ax.scatter(reaction_HF[:,0], reaction_HF[:,1], U_HF, c='b', marker='^', label='HF data')
scatter3 = ax.scatter(reaction_HF_test[:,0], reaction_HF_test[:,1], U_HF_test, c='b', marker='^', label='exact solution')
scatter4 = ax.scatter(reaction_LF_test[:,0], reaction_LF_test[:,1], modelLF.predict(reaction_LF_test), c='b', marker='^', label='estimated $U_{LF}$ by $NN_{LF}$')

# Crea la legenda utilizzando i Path3DCollection
ax.legend(handles=[scatter1, scatter2,scatter3,scatter4])

# Aggiungi etichette agli assi
ax.set_xlabel('mu')
ax.set_ylabel('x')
ax.set_zlabel('U')

# Mostra il grafico
plt.show()

In [ ]:
##########################    SECOND NN: NN_HF    ########################## new
reaction_test_help = modelLF.predict(reaction_HF_test)[:, 0]

reaction_test_in = np.concatenate(
    (reaction_HF_test, reaction_test_help.reshape(-1,1)),axis=1
) # <- TEST INPUT for the second NN: NN_HF

reaction_train_help = modelLF.predict(reaction_HF)[:, 0]  # f_LF(mu_hf_train)
reaction_final = np.concatenate(
    (reaction_HF, reaction_train_help.reshape(-1,1)),axis=1
) # <- TRAINING INPUT for the second NN: NN_HF

name = "3step"

# ----------------------------------------

In [16]:
#########################     TRAIN SET      ##########################
mu_train_LF = np.loadtxt("../DATA_shear_cube/3p/Data_Train_bases/mu_train_LF.txt")
Nlf = np.size(mu_train_LF[:,0])  #number of low-fidelity data to TRAIN the NN

U_lf_train_full = np.loadtxt("../DATA_shear_cube/3p/Data_Train_bases/Ulf_train.txt")
U_lf_train = U_lf_train_full[:,m]

mu_train_HF = np.loadtxt("../DATA_shear_cube/3p/Data_Train_bases/mu_train_HF_" + n_HF_txt)
Nhf = np.size(mu_train_HF)  # number of high-fidelity data to TRAIN the NN

# ADD NOISE

permutation = np.random.permutation(len(mu_train_LF))
mu_train_LF=mu_train_LF[permutation,:][0:10,:]

noise_stddev = np.mean(mu_train_LF,axis=0)*0.1
noise = np.random.normal(0, noise_stddev, np.shape(mu_train_LF))
mu_train_LF=mu_train_LF+noise 

Nlf = np.size(mu_train_LF)  #number of low-fidelity data to TRAIN the NN

U_lf_train_full=U_lf_train_full[permutation][0:10,:]

noise_stddev = 0.005
noise = np.random.normal(0, noise_stddev, np.shape(U_lf_train_full))
U_lf_train_full=U_lf_train_full+noise

U_hf_train = np.loadtxt("../DATA_shear_cube/3p/Data_Train_bases/Uhf_train_" + n_HF_txt)

NepoLF = 4000        # number of epochs for first NN: NN_LF
NepoLin = 1500       # number of epochs for second NN: NN_Lin
NepoHF = 3000        # number of epochs for third NN: NN_HF

In [17]:
#########################     TEST SET      ##########################
mu_test = np.loadtxt("../DATA_shear_cube/3p/Data_Test_bases/mu_test.txt")
N_test = np.size(mu_test)

U_lf_test_full = np.loadtxt("../DATA_shear_cube/3p/Data_Test_bases/Ulf_test.txt")
U_lf_test = U_lf_test_full[:,m]

U_hf_test = np.loadtxt("../DATA_shear_cube/3p/Data_Test_bases/Uhf_test.txt")

In [5]:
##########################    SECOND NN: NN_Lin    ##########################
hf_help = modelLF.predict(mu_train_HF_norm)
hf_lin = np.append(mu_train_HF_norm, hf_help,axis=1)

K.clear_session()

test_help = modelLF.predict(mu_test_norm)

bestLin_params = {'lr' : 0.001, 'kernel_init' : 'glorot_uniform', 'opt' : 'Adam', 'l2weight' : 0.01}
modelLin = getModel(bestLin_params,'Hflin')
histLin = modelLin.fit(hf_lin, U_hf_train, epochs= NepoLin, batch_size = Nhf, verbose = 0)

ULin = modelLin.predict(np.append(mu_test_norm, test_help, axis = 1))



NameError: name 'modelLF' is not defined

In [6]:
##########################    THIRD NN: NN_HF    ##########################
#Input for training NN_HF
hf_help1 = modelLin.predict(hf_lin)
hf_final = np.append(hf_lin, hf_help1, axis=1)

#Input for testing NN_HF
test_help_1 = modelLin.predict(np.append(mu_test_norm, test_help, axis = 1))
test_input_in = np.append(mu_test_norm, np.append(test_help, test_help_1, axis=1), axis=1)

name = '3step'

NameError: name 'modelLin' is not defined

In [7]:
####################    HYPERPARAMETER OPTIMIZATION    #######################
MAX_EVAL = 15

K.clear_session()
bayes_trials = Trials()
opt_list = ['Adam','Adamax']
kernel_list = ['uniform','glorot_uniform']
aux_dic = {'opt': opt_list, 'kernel_init': kernel_list}
space = {
    'nodes' : hp.qloguniform('nodes',np.log(2),np.log(128),2),
     'l2weight' : hp.loguniform('l2weight',np.log(0.0001),np.log(0.1)),
     'lr': hp.loguniform('lr',np.log(0.0001),np.log(0.1)),
     'kernel_init': hp.choice('kernel_init',kernel_list),
     'opt' : hp.choice('opt',opt_list)}

p=3

def objective(params):
    K.clear_session()
    CVres = kCrossVal(p,Nhf,NepoHF,hf_final, U_hf_train,params,name)
    return {'loss' : CVres, 'params':params, 'status': STATUS_OK}

best_params = fmin(fn = objective,
                   space = space,
                   algo = tpe.suggest,
                   max_evals = MAX_EVAL,
                   trials = bayes_trials)
K.clear_session()
transfBestparam(best_params,aux_dic)
finalModel = getModel(best_params,name)

print(best_params)


  0%|          | 0/15 [00:00<?, ?trial/s, best loss=?]

job exception: name 'NepoHF' is not defined



  0%|          | 0/15 [00:00<?, ?trial/s, best loss=?]


NameError: name 'NepoHF' is not defined

In [ ]:
####################    NN_HF training and PREDICTION    #######################
finalModel = getModel(best_params,name)
hist = finalModel.fit(hf_final,U_hf_train,validation_data=(test_input_in, U_hf_test),epochs=NepoHF, batch_size=Nhf, validation_freq=50, verbose=0)

UHF = finalModel.predict(test_input_in)

stop = perf_counter()
elapsed = stop - start
print('Elapsed time: ', elapsed)
print('\nHF Model:')

test_mse = np.mean(np.square(U_hf_test - UHF[:,0]))
print(f"Test MSE: {test_mse}")

r2_HF = 1 - np.sum(np.square(U_hf_test - UHF[:,0])) / np.sum(np.square(U_hf_test - np.mean(U_hf_test)))
print(f"R^2: {r2_HF}")

print('Number of base functions: ', int(Nlf_models[m]))
print('Number of HF data: ', n_HF)

In [ ]:
####################    TRAINING INSIGHTS    #######################
plt.figure()
plt.subplot(2,1,1)
plt.plot(hist.history['mse'],color='red',label='High fidelity train mse')
plt.plot(histLin.history['mse'],color='green',label='high fidelity lin')
plt.plot(histLF.history['mse'],color='black',label='Low fidelity')
plt.legend()
plt.yscale('log')
plt.subplot(2,1,2)
plt.plot(hist.history['val_mse'],color='red')
plt.yscale('log')
